## Dados temperatura sem El Niño.

In [42]:
# Setup — execute uma única vez
import json
import re
import time
import unicodedata
from datetime import datetime, timezone
import pandas as pd
import requests
from pathlib import Path

https://archive-api.open-meteo.com/v1/archive

In [43]:
parametros = {
    "latitude": -1.4558,
    "longitude": -48.4902,
    "start_date": "2023-01-01", # Data inicial
    "end_date": "2023-01-31",   # Data final
    "hourly": "temperature_2m"  # O que você quer medir
    }

In [ ]:
# Cliente HTTP com retry — copie este padrão para qualquer projeto seu
def get_json(url, params=parametros, tentativas=4, espera_base=1.6, timeout=30):
    """
    GET que devolve JSON, com timeout e retry exponencial.

    Trata como transitório: 429 (rate limit) e 5xx (erro do servidor).
    Trata como definitivo: 4xx (o pedido está errado — insistir não resolve).
    """
    cabecalhos = {"Accept": "application/json", "User-Agent": "CESUPA-ETL-Lab/1.0"}


    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, params=params, headers=cabecalhos, timeout=timeout)
        except requests.RequestException as erro:
            if tentativa == tentativas:
                raise
            print(f"  rede falhou ({type(erro).__name__}); tentativa {tentativa}/{tentativas}")
            time.sleep(espera_base ** tentativa)
            continue

        if resposta.status_code == 200:
            return resposta.json()

        if resposta.status_code in (429, 500, 502, 503, 504):
            espera = espera_base ** tentativa
            print(f"  status {resposta.status_code}; aguardando {espera:.1f}s "
                  f"(tentativa {tentativa}/{tentativas})")
            time.sleep(espera)
            continue

        raise RuntimeError(f"falha definitiva {resposta.status_code} em {resposta.url}: "
                           f"{resposta.text[:200]}")

    raise RuntimeError(f"desisti de {url} após {tentativas} tentativas")


def salvar_bronze(dados, nome, fonte, endpoint):
    """
    Bronze = o dado como veio, dentro de um envelope com metadados.

    Nunca sobrescreva a Bronze com dado tratado: ela é a única cópia fiel da origem,
    e é dela que você reprocessa quando descobrir um erro na transformação.
    """
    envelope = {
        "_fonte": fonte,
        "_endpoint": endpoint,
        "_extraido_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        # "_load_id": LOAD_ID,
        "dados": dados
    }
    destino = Path(r"C:\Users\joaof\Downloads\el-nino-energy-effects-brazil-analysis-main\el-nino-energy-effects-brazil-analysis-main") / "bronze" / f"{nome}.json"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(json.dumps(envelope, ensure_ascii=False), encoding="utf-8")
    print(f"  bronze gravada: {destino}  ({destino.stat().st_size / 1024:.1f} KB)")
    return destino

In [45]:
URL = "https://archive-api.open-meteo.com/v1/archive"

openMeteo = get_json(URL, parametros)

print(f"municípios recebidos: {len(openMeteo)}")
print("\nEstrutura do primeiro registro:")
print(json.dumps(openMeteo, ensure_ascii=False, indent=2)[:900])

salvar_bronze(openMeteo, "Open-Meteo - Clima", "Open-Meteo", URL)

municípios recebidos: 9

Estrutura do primeiro registro:
{
  "latitude": -1.4411248,
  "longitude": -48.488922,
  "generationtime_ms": 0.141143798828125,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "elevation": 19.0,
  "hourly_units": {
    "time": "iso8601",
    "temperature_2m": "°C"
  },
  "hourly": {
    "time": [
      "2023-01-01T00:00",
      "2023-01-01T01:00",
      "2023-01-01T02:00",
      "2023-01-01T03:00",
      "2023-01-01T04:00",
      "2023-01-01T05:00",
      "2023-01-01T06:00",
      "2023-01-01T07:00",
      "2023-01-01T08:00",
      "2023-01-01T09:00",
      "2023-01-01T10:00",
      "2023-01-01T11:00",
      "2023-01-01T12:00",
      "2023-01-01T13:00",
      "2023-01-01T14:00",
      "2023-01-01T15:00",
      "2023-01-01T16:00",
      "2023-01-01T17:00",
      "2023-01-01T18:00",
      "2023-01-01T19:00",
      "2023-01-01T20:00",
      "2023-01-01T21:00",
      "2023-01-01T22:00",
  bronze gravada: C:\Users\joaof\Downloads

WindowsPath('C:/Users/joaof/Downloads/el-nino-energy-effects-brazil-analysis-main/el-nino-energy-effects-brazil-analysis-main/bronze/Open-Meteo - Clima.json')